In [1]:
# =============================================================================
#  RevPlan 시뮬레이션 — 배치/스케줄 실행 셀  (수동 실행과 동일 기준)
#  MLWB 노트북 셀에 그대로 붙여넣어 사용.
#  작성 2026-09-09 / 개정 2026-09-09 (수동 실행 기준으로 정렬 + OE 재개 추가)
# =============================================================================
#  ▣ 왜 개정했는가 — 2026-09-08 배치 vs 수동 로그 비교로 확인된 차이
#
#    항목                        배치(구)        수동           →  이 셀
#    ─────────────────────────────────────────────────────────────────────
#    REVPLAN_MEASURED_LT         OFF (dark)     ON                 "1"
#    REVPLAN_EXCLUDE_HOLD_LOTS   OFF (KEPT)     ON                 "1"
#    REVPLAN_COP_USAGE           OFF            ACTIVE             "1"
#    max_delay_days              200            90                 90
#    ─────────────────────────────────────────────────────────────────────
#    실행 시간                   25분 48초      9분 54초
#    allocation 행수             582,967        597,657
#
#    * max_delay_days: 엔진은 배분 실패 시 일 단위로 스캔한다. 구 셀이 이 값을
#      지정하지 않아 SimulationParams 기본값 200이 들어갔고, 90 대비 2.2배를
#      훑어 실행 시간이 2.6배가 됐다. → 느림의 주원인.
#    * EXCLUDE_HOLD_LOTS: HOLD lot 7,823개(874,027,963 EA)를 스케줄에서 뺄지.
#      배치는 "생산 가능"으로, 수동은 "불가"로 처리해 매출이 크게 벌어졌다.
#    * MEASURED_LT: 외주 통과 스텝(전체 배분 스텝의 44.5%)의 소요 시간을 실측
#      중위값으로 쓸지 PlanLt로 쓸지. 리드타임이 달라져 배분 결과가 바뀐다.
#    * Net Demand 3개 값(계획 / 재고차감 / Net)은 두 실행에서 완전히 동일했다.
#      → 차이는 전부 배분 단계에서 발생. 3번 요건(Net Demand)은 재현된다.
#
#  ⚠️ 완전히 같은 숫자는 나오지 않는다
#    설정을 맞춰도 (1) o_custom_WIP은 시간별 스냅샷, PK1_URGENCY_WIP_LOT은 스냅샷
#    필터 없이 전량을 읽으므로 실행 시각이 다르면 입력이 달라지고, (2) 엔진이
#    비결정적이다 — allocation_engine 452/632행이 lot 목록을 set으로 만들고
#    _select_lots_for_demand가 remaining_steps 단일 키로만 정렬하므로 동점 lot의
#    순서가 프로세스마다 바뀐다(PYTHONHASHSEED 미고정).
#    실측: 입력·플래그가 완전히 동일한 두 실행에서 allocation 604,559 vs 599,252
#    (0.88% 차이). 따라서 전후 비교는 매출/행수가 아니라 Net Demand로 하라.
# =============================================================================

# -----------------------------------------------------------------------------
# STEP 1. 부트스트랩 — revplan_engine 상위 폴더를 sys.path에 넣는다 (필수)
#   2026-09-09 스케줄 첫 실행이 ModuleNotFoundError로 죽은 원인이 이 셀의 부재였다.
#   성공한 수동 노트북의 [5]번 셀과 동일한 로직.
#   실제 경로: cwd=/home/jovyan/SM WIP 260629/revplan_engine
#             engine parent=/home/jovyan/SM WIP 260629
# -----------------------------------------------------------------------------
import os, sys, time
from datetime import datetime, timedelta, date

_T0 = time.time()


def _find_engine_parent():
    """revplan_engine/run_simulation.py 를 담고 있는 폴더를 찾는다.
    papermill / 스케줄러의 cwd는 예측할 수 없으므로 후보를 순서대로 훑는다.
    ENGINE_PARENT 환경변수로 강제 지정 가능."""
    cands = []
    env = os.environ.get("ENGINE_PARENT")
    if env:
        cands.append(env)
    d = os.getcwd()
    if os.path.basename(d) == "revplan_engine" and os.path.isfile(
            os.path.join(d, "run_simulation.py")):
        cands.append(os.path.dirname(d))          # 노트북이 패키지 안에 있는 경우
    for _ in range(6):                            # cwd + 상위 6단계
        cands.append(d)
        d = os.path.dirname(d)
    cands += [os.path.expanduser("~"), "/home/jovyan"]
    for base in cands:
        if base and os.path.isfile(os.path.join(base, "revplan_engine", "run_simulation.py")):
            return base
    home = os.path.expanduser("~")                # 최후: home 아래 제한 탐색
    for root, dirs, _ in os.walk(home):
        if root[len(home):].count(os.sep) > 4:
            dirs[:] = []
            continue
        dirs[:] = [x for x in dirs if not x.startswith(".")
                   and x not in ("node_modules", "__pycache__", "site-packages")]
        if os.path.isfile(os.path.join(root, "revplan_engine", "run_simulation.py")):
            return root
    return None


print("cwd =", os.getcwd())
_parent = _find_engine_parent()
if _parent is None:
    raise ModuleNotFoundError(
        "revplan_engine not found. 이 노트북을 revplan_engine 폴더와 같은 레벨(또는 그 안)에 "
        "두거나 ENGINE_PARENT 환경변수를 지정하세요. 경로 확인: "
        "!find ~ -maxdepth 5 -name run_simulation.py")
if _parent not in sys.path:
    sys.path.insert(0, _parent)
print("engine parent =", _parent)

# -----------------------------------------------------------------------------
# STEP 2. 실행 플래그 — 수동 실행과 동일하게 맞춘다
#   ⚠️ revplan_engine을 import하기 전에 세팅해야 한다. REVPLAN_SCHEMA_MIGRATE는
#      celonis_io 모듈 최상단에서 한 번만 읽히므로, 이미 celonis_io를 import한
#      커널에서 값을 바꿔도 반영되지 않는다. 새 커널에서 이 셀을 먼저 돌릴 것.
# -----------------------------------------------------------------------------
# 외주 통과 스텝(배분 스텝의 44.5%)의 소요 시간을 WipHistory/WIP 실측 중위값으로.
# 2026-07-30 회의 방향: 표준(PlanLt)보다 실측 우선. 수동 실행이 ON이었다.
os.environ["REVPLAN_MEASURED_LT"] = "1"

# HOLD lot(STATE=HOLD / WIPHOLD=Y)을 스케줄에서 제외. 수동 실행이 ON이었다.
# OFF면 7,823 lot / 874,027,963 EA를 "생산 가능"으로 잡아 낙관 편향이 생긴다.
os.environ["REVPLAN_EXCLUDE_HOLD_LOTS"] = "1"

# CopClass 소요량 집계 기준(A→SEQ2, B→M510N, C→M520N, D 제외). SimStockMaster에만
# 영향하고 배분·매출에는 영향 없다. 수동 실행이 ACTIVE였으므로 맞춘다.
os.environ["REVPLAN_COP_USAGE"] = "1"

# ▣ 이력 보존 (2026-09-09 결정) — 모든 SIM_* 테이블의 실행 이력을 남긴다.
#   🔴 절대 os.environ["REVPLAN_SCHEMA_MIGRATE"] = "..." 로 값을 세팅하지 말 것.
#      이 변수에 이름이 든 테이블은 매 실행 삭제·재생성되어 과거 실행이 사라진다.
#      (2026-09-09 사고: 이 셀이 ""로 세팅했는데도 이전 데이터가 다 지워졌다.
#       원인 — celonis_io는 2694행에서 import 시점에 딱 한 번 이 변수를 읽는다.
#       이미 celonis_io가 로드된 커널에서는 여기서 ""로 바꿔도 반영되지 않고,
#       그 커널이 들고 있던 옛 값 'allocation,WIPMaster'가 그대로 살아 삭제됐다.)
#   ✅ 해법: 값을 "세팅"하지 말고 "제거"만 한다. celonis_io의 코드 기본값이 이미
#      ""(이력 보존)이므로, 변수가 없으면 안전하게 누적된다. pop은 이전 셀/커널이
#      남긴 위험한 값도 없애준다. 단 pop도 "이미 로드된 celonis_io"에는 소급되지
#      않으므로, 아래 커널 재시작 확인이 진짜 안전장치다.
os.environ.pop("REVPLAN_SCHEMA_MIGRATE", None)

# ⚠️ CELONIS_OUTPUT_RESET은 절대 세팅하지 말 것 — 21개 테이블 전부를 날린다. 제거만.
os.environ.pop("CELONIS_OUTPUT_RESET", None)

# 🔴 커널 재시작 확인 — celonis_io가 이미 로드돼 있으면 위 pop이 소급 안 된다.
#    이 셀이 "그 커널의 첫 실행"인지 검증한다. celonis_io가 이미 sys.modules에
#    있으면(=이전에 돌린 커널) 즉시 멈추고 "Restart Kernel" 후 재실행하게 한다.
#    이게 과거 데이터 삭제를 막는 진짜 방어선이다.
_already_loaded = [m for m in sys.modules
                   if m == "revplan_engine.celonis_io" or m.endswith(".celonis_io")]
if _already_loaded:
    raise RuntimeError(
        "🔴 celonis_io가 이미 이 커널에 로드되어 있습니다 "
        f"({_already_loaded}).\n"
        "   REVPLAN_SCHEMA_MIGRATE는 import 시점에 한 번만 읽히므로, 이 상태로\n"
        "   실행하면 이전 커널이 들고 있던 옛 값 때문에 과거 SIM_* 데이터가\n"
        "   삭제될 수 있습니다.\n"
        "   → Kernel ▸ Restart Kernel and Clear Outputs 후 이 셀을 첫 셀로 실행하세요.")

# 배분 실패 시 일 단위 스캔 범위. 200이면 90 대비 2.2배를 훑어 실행 시간이
# 2.6배가 된다(실측: 25분 48초 vs 9분 54초). 수동 실행 기준은 90.
MAX_DELAY_DAYS = 90

# 이동계획 리비전. None이면 리비전 명칭의 사전순 최신값으로 폴백한다.
#   ⚠️ 이는 "최종 릴리즈"가 아니라 "가장 최근 생성"이다. o_custom_MovePlan에
#      릴리즈 상태 컬럼이 없어 확정 여부를 판별할 수 없다. 확정본만 쓰려면 명시 지정.
REVENUE_PLAN_ID = None

# 시작월(YYYYMM). 현재 월로 두면 그 달 출하 실적(MTD)이 있어 Net Demand의 출하
# 차감이 동작한다. 미래 월이면 차감이 0이 된다(현 운영 방식). 현업이 출하 차감
# 기준을 정하기 전에는 수동 실행과 동일하게 유지한다.
START_MONTH = "202610"

# -----------------------------------------------------------------------------
# STEP 3. 자격증명 사전 점검
#   성공한 수동 실행은 'auth: explicit key' 경로였다 → REVPLAN_USER_KEY 환경변수
#   또는 ~/.revplan_user_key 파일이 있었다는 뜻. 스케줄러 컨테이너에 없으면
#   MLWB zero-config로 떨어지고 데이터 권한이 없어 읽기가 403이 될 수 있다.
# -----------------------------------------------------------------------------
_keyfile = os.path.expanduser("~/.revplan_user_key")
_has_key = bool(os.environ.get("REVPLAN_USER_KEY")) or os.path.isfile(_keyfile)
print(f"[auth] explicit key: {'있음' if _has_key else '없음 → MLWB zero-config로 시도됨'}")
if not _has_key:
    print("[auth] ⚠ zero-config 앱키는 데이터 권한이 없어 읽기가 403이 될 수 있습니다.")
    print(f"[auth]   해결: REVPLAN_USER_KEY 환경변수 또는 {_keyfile} 파일")

# -----------------------------------------------------------------------------
# STEP 4. 실행 식별자
#   OE가 dpInstanceId를 주입했으면 그것을 쓴다(프론트/OE와 correlate 가능).
#   없으면 KST 기준 고유 id를 만든다 — celonis_io.write_outputs가
#   {"", "local-test", "none", "null", "mlwb_run"} 이면 SIM_* 오염 방지를 위해
#   아무것도 쓰지 않기 때문이다(2026-09-09 배치가 전량 스킵된 원인).
# -----------------------------------------------------------------------------
_PLACEHOLDERS = {"", "local-test", "none", "null", "mlwb_run"}
DP_INSTANCE_ID = None
for _k in ("dpInstanceId", "dpinstanceid", "dp_instance_id", "instanceId", "simulationId"):
    _v = globals().get(_k)
    if _v and str(_v).strip().lower() not in _PLACEHOLDERS:
        DP_INSTANCE_ID = str(_v).strip()
        break

KST = datetime.utcnow() + timedelta(hours=9)      # 컨테이너가 UTC로 동작 → +9
SIMULATION_ID = DP_INSTANCE_ID or f"SCHED_{KST.strftime('%Y%m%d_%H%M')}"
_ID_SRC = "OE 주입(dpInstanceId)" if DP_INSTANCE_ID else "자동 생성(스케줄)"

print("=" * 78)
print(f" 배치 실행 시작  KST {KST.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   simulation_id   : {SIMULATION_ID}   ({_ID_SRC})")
print(f"   start_month     : {START_MONTH}")
print(f"   max_delay_days  : {MAX_DELAY_DAYS}")
print(f"   revenue_plan_id : {REVENUE_PLAN_ID or '(미지정 → 사전순 최신으로 폴백)'}")
print(f"   MEASURED_LT     : {os.environ.get('REVPLAN_MEASURED_LT')}")
print(f"   EXCLUDE_HOLD    : {os.environ.get('REVPLAN_EXCLUDE_HOLD_LOTS')}")
print(f"   COP_USAGE       : {os.environ.get('REVPLAN_COP_USAGE')}")
print(f"   SCHEMA_MIGRATE  : {os.environ.get('REVPLAN_SCHEMA_MIGRATE', '')!r}  (빈 값 = 이력 보존)")
print("=" * 78)

# -----------------------------------------------------------------------------
# STEP 5. 실행
# -----------------------------------------------------------------------------
from revplan_engine.run_simulation import SimulationParams, main

params = SimulationParams(
    simulation_id   = SIMULATION_ID,
    simulation_name = None,              # None → run_simulation이 KST 기준 자동 생성
    revenue_plan_id = REVENUE_PLAN_ID,
    start_month     = START_MONTH,
    start_date      = date.today(),
    max_delay_days  = MAX_DELAY_DAYS,    # ← 구 셀에 없어 기본값 200이 들어갔다
)

results = main(params=params)

# -----------------------------------------------------------------------------
# STEP 6. 저장 성공 확인
# -----------------------------------------------------------------------------
_ELAPSED = time.time() - _T0
print("\n" + "=" * 78)
print(f" 실행 요약   소요 {_ELAPSED/60:.1f}분")
print("=" * 78)

rt = results.get("run_tracker")
if rt is not None and rt.height:
    r = rt.row(0, named=True)
    print(f"   status          : {r.get('status')}")
    print(f"   simulation_id   : {r.get('simulation_id')}")
    print(f"   simulation_name : {r.get('simulation_name')}")   # 자동 생성된 이름
    print(f"   revenue_plan_id : {r.get('revenue_plan_id')}")   # 실제 사용된 리비전
    print(f"   run_id          : {r.get('run_id')}")
    print(f"   run_timestamp   : {r.get('run_timestamp')} (UTC)")

print("\n   테이블별 행 수:")
for name, df in results.items():
    h = getattr(df, "height", 0)
    print(f"     {'⚠️ ' if h == 0 else '   '}{name:<36} {h:>10,}")

# -----------------------------------------------------------------------------
# STEP 7. OE 재개 이벤트 (OE가 트리거한 경우에만)
#   OE가 시작한 인스턴스는 이 이벤트를 받아야 'finish-simulation' 재개 스텝을
#   지나 다음 단계로 넘어간다. 보내지 않으면 OE가 계속 대기한다.
#   스케줄이 스스로 시작한 실행은 재개할 인스턴스가 없으므로 건너뛴다.
#   필요 환경변수: oe_client_id / oe_client_secret / OE_PACKAGE_KEY
#                 (+ CELONIS_BASE_URL 또는 CELONIS_URL)
# -----------------------------------------------------------------------------
if DP_INSTANCE_ID:
    try:
        from revplan_engine.orchestration import emit_finished_simulation
        emit_finished_simulation(DP_INSTANCE_ID)
    except Exception as _ex:            # noqa: BLE001 — 저장은 이미 끝났다
        print(f"[oe][ERROR] 재개 이벤트 실패 ({type(_ex).__name__}: {_ex})")
        print("[oe]   → OE 인스턴스가 대기 상태로 남습니다. "
              "oe_client_id / oe_client_secret / OE_PACKAGE_KEY 설정을 확인하세요.")
else:
    print("\n[oe] dpInstanceId 없음 — OE 재개 이벤트 건너뜀 "
          "(스케줄 자체 실행이라 재개할 OE 인스턴스가 없습니다).")

# -----------------------------------------------------------------------------
# STEP 8. 로그 점검 체크리스트
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print(" ⚠️ 로그에서 반드시 확인할 것")
print("=" * 78)
print("   1) '⛔ SKIP write_outputs' 가 없어야 한다 (있으면 아무것도 저장되지 않음)")
print("   2) 'appended SIM_allocation (+N rows)' — created가 아니라 appended여야")
print("      이력이 쌓인다. created면 REVPLAN_SCHEMA_MIGRATE가 어딘가에서 세팅된 것.")
print("   3) 'measured LT installed: … ACTIVE (REVPLAN_MEASURED_LT=1)'")
print("   4) 'HOLD 제외: 7,823 lots … excluded' — KEPT가 아니어야 수동과 같은 기준")
print("   5) 'max_delay_days=90' — 200이면 실행 시간이 2.6배가 된다")
print("   6) 'Revenue plan: N rows' == 'Grouping_model-month combinations: N'")
print("      → 같아야 정상. 다르면 Net Demand 집계 단위가 바뀐 회귀다.")
print("   7) 'snapshot looks HALF-WRITTEN' 가 나오면 00시 배치가 자정 적재와 겹친 것")
print("      → 배치 시각을 00:30 정도로 늦추는 편이 안전하다.")
print("\n   ※ 전후 비교는 매출/행수가 아니라 Net Demand 3개 값으로 하라.")
print("     (계획 수량 / 재고 차감 / Net Demand — 이 셋만 결정적이다)")
print("=" * 78)

cwd = /home/jovyan/SM WIP 260629/revplan_engine
engine parent = /home/jovyan/SM WIP 260629
[auth] explicit key: 있음
 배치 실행 시작  KST 2026-09-09 04:16:15
   simulation_id   : SCHED_20260909_0416   (자동 생성(스케줄))
   start_month     : 202610
   max_delay_days  : 90
   revenue_plan_id : (미지정 → 사전순 최신으로 폴백)
   MEASURED_LT     : 1
   EXCLUDE_HOLD    : 1
   COP_USAGE       : 1
   SCHEMA_MIGRATE  : ''  (빈 값 = 이력 보존)
=== Celonis read_inputs ===


/tmp/ipykernel_20736/1614865257.py:184: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  KST = datetime.utcnow() + timedelta(hours=9)      # 컨테이너가 UTC로 동작 → +9
[2026-09-08 19:16:17,681] WARNING: Your PyCelonis Version 2.14.2 is outdated (Newest Version: 2.16.1). Please upgrade the package via: pip install --extra-index-url=https://pypi.celonis.cloud/ pycelonis pycelonis_core --upgrade


   ✓ auth: explicit key -> https://lg-innotek.eu-1.celonis.cloud (key_type=USER_KEY)
[2026-09-08 19:16:17,749] INFO: Initial connect successful! PyCelonis Version: 2.14.2
[2026-09-08 19:16:17,912] INFO: `ml-workbench` permissions: ['CREATE_APPS', 'USE_ALL_APPS', 'MANAGE_ALL_APPS', 'CREATE_WORKSPACES', 'MANAGE_ALL_WORKSPACES', 'VIEW_CONFIGURATION']
[2026-09-08 19:16:17,913] INFO: `team` permissions: ['MANAGE_AUDIT_LOGS', 'MANAGE_SSO_SETTINGS', 'USE_AUDIT_LOGS_API', 'MANAGE_ADOPTION_VIEWS', 'MANAGE_GENERAL_SETTINGS', 'MANAGE_GROUPS', 'MANAGE_APPLICATIONS', 'MANAGE_ON_PREM_CLIENTS', 'USE_STUDIO_ADOPTION_API', 'MANAGE_LOGIN_HISTORY', 'MANAGE_LICENSE_SETTINGS', 'USE_LOGIN_HISTORY_API', 'USE_USER_GROUP_INFO_API', 'MANAGE_MEMBERS', 'MANAGE_UPLINK_INTEGRATIONS', 'MANAGE_PERMISSIONS', 'MANAGE_ADMIN_NOTIFICATIONS', 'MANAGE_DOWNLOAD_PORTAL', 'IMPORT_MEMBERS']
[2026-09-08 19:16:17,913] INFO: `process-repository` permissions: ['CREATE_AND_MODIFY_CATEGORIES', 'USE_CATEGORIES', 'DELETE_EXISTING_CATEG

0it [00:00, ?it/s]

[2026-09-08 19:16:21,585] INFO: Export result chunks for data export with id 'b5ac3212-b244-4681-9ea3-e35a64c4edb7'
[2026-09-08 19:16:26,700] INFO: Successfully created data export using api v1 with id 'cd080010-8cae-4edf-b860-3659d075d25e'
[2026-09-08 19:16:26,701] INFO: Wait for execution of data export with id 'cd080010-8cae-4edf-b860-3659d075d25e'


...

[2026-09-08 19:31:05,946] INFO: Successfully created data push job with id '571464bb-9576-4bbf-b374-36658216aee9'
[2026-09-08 19:31:05,947] INFO: Add data frame as file chunks to data push job with id '571464bb-9576-4bbf-b374-36658216aee9'


  0%|          | 0/1 [00:00<?, ?it/s]

[2026-09-08 19:31:06,065] INFO: Successfully upserted file chunk to data push job with id '571464bb-9576-4bbf-b374-36658216aee9'
[2026-09-08 19:31:06,343] INFO: Successfully triggered execution for data push job with id '571464bb-9576-4bbf-b374-36658216aee9'
[2026-09-08 19:31:06,343] INFO: Wait for execution of data push job with id '571464bb-9576-4bbf-b374-36658216aee9'


0it [00:00, ?it/s]

[2026-09-08 19:31:12,129] INFO: Successfully deleted data push job with id '571464bb-9576-4bbf-b374-36658216aee9'
[2026-09-08 19:31:12,130] INFO: Successfully appended rows to table 'SIM_uncommitted_demand_by_lot' in data pool
   ✓ appended SIM_uncommitted_demand_by_lot  (+1063 rows)

 실행 요약   소요 14.9분
   status          : SUCCESS
   simulation_id   : SCHED_20260909_0416
   simulation_name : 20260909_MP202609-01W-001_0416
   revenue_plan_id : MP202609-01W-001
   run_id          : rev_alloc_20260908_191657_115e3765
   run_timestamp   : 2026-09-08 19:16:57.410517 (UTC)

   테이블별 행 수:
        allocation                              597,100
        WIPMaster                               597,100
        StockMaster                              11,499
        new_lots_created                         15,939
     ⚠️ unrouted_model_demand                         0
        failed_allocations                       15,385
        run_tracker                                   1
        monthly_fulf